# 09 固化实验规范

阶段 4～8 已经完成 Pretraining、SFT、LoRA 和 Distillation。阶段 9 整理这些实验共同使用的记录方法，使一次运行的输入、过程、产物和结论可以相互对应。

```text
实验配置 → 执行命令 → 指标与模型产物 → 运行清单 → 固定评估 → 两次结果比较
```

本阶段沿用 MiniMind 的现有训练与评估入口。后续 Qwen 实验使用 TRL、Transformers、Accelerate 和 TensorBoard，并沿用本阶段确定的记录字段和比较规则。

## 阶段 10 会怎样导入这四个库

本 Notebook 的实际任务是复现 MiniMind 已有结果，因此下面的代码单元只导入 `torch` 和项目内记录函数，不会导入尚未安装、也尚未使用的 Qwen 训练库。阶段 10 安装环境后，四个库的典型入口如下：

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer
from accelerate import Accelerator
from torch.utils.tensorboard import SummaryWriter
```

`AutoModelForCausalLM` 和 `AutoTokenizer` 加载模型与 tokenizer；`SFTTrainer` 组织 SFT；`Accelerator` 管理 device、mixed precision 和多卡启动；`SummaryWriter` 把 loss、learning rate 等 scalar 写入 TensorBoard event。它们会在阶段 10 的训练脚本中实际运行。

In [1]:
# 加载阶段 9 的固定复现配置。
import json
from pathlib import Path

import torch

from llm_learning.experiment_record import (
    compare_reproduction_results,
    scalar_series,
    validate_manifest,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
STAGE_DIR = ROOT / "docs/stages/09_experiment_reproducibility"
reproduction = json.loads(
    (STAGE_DIR / "configs/minimind_reproduction.json").read_text()
)
print(f"profile: {reproduction['profile']}")
print(f"checkpoint: {reproduction['checkpoint']}")
print(f"fixed runs: {len(reproduction['run_outputs'])}")

profile: stage9_minimind_reproduction
checkpoint: checkpoints/minimind/stage5/mini64/weights_step_56480.pth
fixed runs: 2


## 1. 一次实验需要哪三类文件

实验开始前读取 config，运行过程中生成 run manifest，运行结束后写出 result。三类文件的职责如下。

| 文件 | 含义 | 典型内容 |
| --- | --- | --- |
| config | 计划使用的实验配置 | model、dataset、seed、batch size、learning rate、evaluation |
| run manifest | 本次运行的实际记录 | Git commit、GPU、执行命令、输入文件、产物路径 |
| result | 本次运行得到的结果 | loss、perplexity、task metrics、generation |

同一份 config 可以执行多次。每次执行分别生成一份 run manifest 和 result，因此可以追溯某个结果来自哪次运行。

In [2]:
# 检查运行清单模板是否覆盖固定字段。
manifest_template = json.loads(
    (STAGE_DIR / "templates/run_manifest.json").read_text()
)
validate_manifest(manifest_template)
print("manifest fields: " + ", ".join(sorted(manifest_template)))
print(f"seed: {manifest_template['seed']}")
print(f"device: {manifest_template['hardware']['device']}")

manifest fields: artifacts, command, dataset, evaluation, experiment, git_commit, hardware, model, seed, training
seed: 42
device: cuda:0


## 2. JSONL 与 TensorBoard 的关系

JSONL 是每行一个 JSON object 的文本格式。训练程序每隔若干 optimizer step 追加一条记录，例如 `{"optimizer_step": 10, "loss": 1.25}`。下一次记录可能是 `{"optimizer_step": 20, "loss": 1.10}`。

把这些 `step → loss` 的数值对按顺序连接，就得到 loss 曲线。`loss`、`learning_rate` 这类每个 step 只有一个数的指标，TensorBoard 称为 scalar；这个名称只说明它能作为一条曲线的纵轴。

TensorBoard event 保存这些按 step 排列的指标。TensorBoard 读取 event 后，显示 loss、learning rate 等曲线。

```text
metrics.jsonl ─→ 提取 step、指标名称和数值 ─→ TensorBoard event ─→ 曲线
```

JSONL 便于直接检查和后续分析；TensorBoard event 用于交互查看曲线。本阶段从已有 JSONL 生成 event，不重新训练模型。

In [3]:
# 将两条 JSONL 风格记录展开为 TensorBoard scalar。
example_metrics = [
    {"optimizer_step": 1, "loss": 1.25, "learning_rate": 1e-5},
    {"optimizer_step": 2, "loss": 1.10, "learning_rate": 9e-6},
]
print("step  metric          value")
for step, metric, value in scalar_series(example_metrics):
    formatted = f"{value:.2e}" if abs(value) < 1e-3 else f"{value:.4f}"
    print(f"{step:>4}  {metric:<14} {formatted:>10}")

step  metric          value
   1  loss               1.2500
   1  learning_rate    1.00e-05
   2  loss               1.1000
   2  learning_rate    9.00e-06


## 3. 模型权重与训练 checkpoint

模型权重文件保存模型在 forward 中使用的权重。它可以直接用于推理，或作为一次新训练的初始权重。

若要从中断处继续同一次训练，还需要恢复训练进度和 optimizer 的内部状态；这些内容与模型权重一起保存在训练 checkpoint 中：

- optimizer state：恢复 AdamW 的一阶、二阶动量；
- mixed-precision scaler：恢复 FP16 gradient scaling 的当前尺度；
- training state：恢复 optimizer step、epoch 和下一个 batch 的位置；
- RNG state：恢复 random number generator 的状态，使数据打乱、dropout 等随机过程从原位置继续。

阶段 5 的 `weights_step_56480.pth` 是模型权重，`latest.pt` 是可恢复训练的 checkpoint。下面查看 `latest.pt` 的顶层字段。

In [4]:
# 查看阶段 5 可恢复 checkpoint 的组成。
checkpoint = torch.load(
    ROOT / "checkpoints/minimind/stage5/mini64/latest.pt",
    map_location="cpu",
    weights_only=False,
)
state = checkpoint["training_state"]
print("checkpoint fields: " + ", ".join(sorted(checkpoint)))
print(f"optimizer step: {state['optimizer_step']}")
print(f"next batch after resume: {state['next_batch']} (first batch of the next epoch)")
print(f"stop reason: {state['stop_reason']}")

checkpoint fields: config, data_manifest, history, model, optimizer, rng_state, scaler, tokenizer, training_state
optimizer step: 56480
next batch after resume: 0 (first batch of the next epoch)
stop reason: stop_after_step


## 4. 本阶段怎样判断复现成功

本阶段连续运行两次相同的固定评估。两次运行使用相同的 Git commit、模型 checkpoint、validation row IDs、generation config、seed、device 和 dtype。

固定评估产生 validation loss、perplexity 和 greedy generation。比较规则如下：

- validation loss 与 perplexity 的绝对差不大于 $10^{-6}$；
- greedy generation 的 token IDs 完全一致；
- wall time 和 tokens/s 作为运行速度记录，不参与复现结论。

greedy decoding 每一步选择 logit 最大的 token。输入、模型和计算条件固定时，可以直接比较两次生成的 token IDs。

In [5]:
# 用一个小例子展示数值容差和精确字段。
first = {"validation_loss": 1.0, "greedy_token_ids": [[4, 2]]}
second = {"validation_loss": 1.0000004, "greedy_token_ids": [[4, 2]]}
comparison = compare_reproduction_results(
    first,
    second,
    numeric_fields=["validation_loss"],
    exact_fields=["greedy_token_ids"],
    absolute_tolerance=1e-6,
)
difference = comparison['numeric']['validation_loss']['difference']
print(f"loss difference: {difference:.7f}")
print(f"greedy tokens match: {comparison['exact']['greedy_token_ids']['matches']}")
print(f"reproduced: {comparison['reproduced']}")

loss difference: 0.0000004
greedy tokens match: True
reproduced: True


## 5. 实际复现顺序

阶段 README 给出完整命令。各步骤的输入和输出如下：

```text
同一 config ─→ 固定评估 run 1 ─→ run_1.json + manifest_1.json
           └→ 固定评估 run 2 ─→ run_2.json + manifest_2.json
run_1.json + run_2.json ─→ comparison.json
已有 metrics.jsonl ─→ TensorBoard event
运行结果与比较结论 ─→ RESULTS.md
```

固定评估调用 `model.eval()` 和 `torch.no_grad()`，不会执行 backward 或 optimizer step，模型 parameters 保持不变。

## 6. 本次固定评估结果

本次使用阶段 5 的 MiniMind 64M Full SFT checkpoint，对同一组 2,048 条 validation rows 连续评估两次。每次评估处理 852,275 个 validation tokens。

两次运行的 validation loss 与 perplexity 完全一致，两个 greedy generation 的 token IDs 也完全一致。wall time 分别为约 20.67 秒和 20.05 秒，只用于记录运行速度。

In [6]:
# 读取两次固定评估与比较结果，不重复运行 evaluation。
result_dir = ROOT / "outputs/minimind/stage9/reproduction"
run_1 = json.loads((result_dir / "run_1.json").read_text())
run_2 = json.loads((result_dir / "run_2.json").read_text())
actual_comparison = json.loads((result_dir / "comparison.json").read_text())

print("metric                 run 1          run 2      difference")
for metric in ("validation_loss", "perplexity"):
    difference = actual_comparison["numeric"][metric]["difference"]
    label = metric.replace("_", " ")
    print(f"{label:<18} {run_1[metric]:>12.10f} {run_2[metric]:>14.10f} {difference:>15.10f}")
print(f"{'validation tokens':<18} {run_1['validation_tokens']:>12} {run_2['validation_tokens']:>14} {0:>15}")
print(f"greedy token IDs match: {actual_comparison['exact']['greedy_token_ids']['matches']}")
print(f"reproduced: {actual_comparison['reproduced']}")
event_dir = ROOT / "outputs/tensorboard/stage8_ce_kd"
tensorboard_events = list(event_dir.glob("events.out.tfevents.*"))
print(f"stage8_ce_kd TensorBoard events: {len(tensorboard_events)}")

metric                 run 1          run 2      difference
validation loss     1.5630904553   1.5630904553    0.0000000000
perplexity          4.7735509186   4.7735509186    0.0000000000
validation tokens        852275         852275               0
greedy token IDs match: True
reproduced: True
stage8_ce_kd TensorBoard events: 1


## 7. 结果不一致时先判断哪一类变化

| 类别 | 示例 | 处理方式 |
| --- | --- | --- |
| 算法变化 | CE 改成 KD | 作为新实验，不与原实验混为复现 |
| 数据变化 | validation row IDs 改变 | 核对 split 和数据 revision |
| 模型变化 | checkpoint 或 tokenizer 改变 | 记录并核对模型身份 |
| 随机过程变化 | sampling token IDs 改变 | 核对 seed 和 generation config |
| 实现错误 | greedy token IDs 在固定条件下改变 | 检查代码、train/eval mode、dtype 和输入 |

检查顺序从输入条件开始：先核对 commit、checkpoint、数据和配置，再比较评估结果。这样可以区分实验条件发生变化和相同条件下的结果差异。

## 8. Qwen 阶段如何复用

阶段 10 使用 `configs/qwen_sft_template.yaml` 作为 TRL CLI 配置起点。当前模板声明 model、dataset、seed、batch size、训练长度、评估频率、保存频率和 TensorBoard。训练运行后产生 checkpoint；从 checkpoint 恢复时，再在运行命令或训练配置中指定其路径。TRL 执行 SFT，Transformers Trainer 管理训练与 checkpoint，Accelerate 负责 device 和 mixed precision。

更换模型和训练框架后，config 的具体字段会变化；run manifest 中的模型、数据、命令、环境和产物字段继续保留。复现检查仍然固定输入条件，再比较数值指标和生成结果。